# Sentiment Analysis σε IMDb Reviews: Προεπεξεργασία & Αναπαράσταση Κειμένου

Στο παρόν notebook υλοποιείται pipeline προετοιμασίας κειμένου για **sentiment classification** σε reviews του IMDb.
Η διαδικασία περιλαμβάνει:

- έλεγχο/ποσοτικοποίηση θορύβου στο raw κείμενο (URLs, HTML tags, ειδικοί χαρακτήρες κ.λπ.)
- δημιουργία καθαρισμένης αναπαράστασης (`clean_review`)
- μετατροπή κειμένου σε χαρακτηριστικά με **TF-IDF**
- εκπαίδευση **Word2Vec** embeddings και δημιουργία document-level vectors (μέσος όρος λέξεων)
- (προαιρετικά) μείωση διάστασης με **PCA** για αποδοτικότερη εκπαίδευση/οπτικοποίηση

Στόχος είναι να συγκριθούν αναπαραστάσεις (TF-IDF vs Word2Vec) ως είσοδος σε downstream μοντέλο ταξινόμησης.

## 1. Φόρτωση δεδομένων & αρχικός έλεγχος

Το dataset φορτώνεται τοπικά από το project directory.
Χρησιμοποιείται `encoding="latin1"` ώστε να αποφευχθούν προβλήματα αποκωδικοποίησης χαρακτήρων που εμφανίζονται συχνά σε αρχεία κειμένου.
Στη συνέχεια ελέγχεται το μέγεθος του dataset και γίνεται μια πρώτη επιθεώρηση των πεδίων.

In [50]:
import pandas as pd
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_validate
from xgboost import XGBClassifier


os.chdir(r"C:\Users\Mkasi\Downloads\PROJECTS\ProjectChalkidi")

df = pd.read_csv(
    r"C:\Users\Mkasi\Downloads\PROJECTS\ProjectChalkidi\imdb_master.csv",
    encoding="latin1")

print(df.shape)
df.head()

(100000, 5)


,Unnamed: 0,type,review,label,file
0,0,test,Once again Mr. Costner has dragged out a movie...,neg,0_2.txt
1,1,test,This is an example of why the majority of acti...,neg,10000_4.txt
2,2,test,"First of all I hate those moronic rappers, who...",neg,10001_1.txt
3,3,test,Not even the Beatles could write songs everyon...,neg,10002_3.txt
4,4,test,Brass pictures (movies is not a fitting word f...,neg,10003_3.txt


Remove HTML tags, URLs, non-alphanumeric characters & Convert to lowercase

In [51]:
df['review'].iloc[3]

"Not even the Beatles could write songs everyone liked, and although Walter Hill is no mop-top he's second to none when it comes to thought provoking action movies. The nineties came and social platforms were changing in music and film, the emergence of the Rapper turned movie star was in full swing, the acting took a back seat to each man's overpowering regional accent and transparent acting. This was one of the many ice-t movies i saw as a kid and loved, only to watch them later and cringe. Bill Paxton and William Sadler are firemen with basic lives until a burning building tenant about to go up in flames hands over a map with gold implications. I hand it to Walter for quickly and neatly setting up the main characters and location. But i fault everyone involved for turning out Lame-o performances. Ice-t and cube must have been red hot at this time, and while I've enjoyed both their careers as rappers, in my opinion they fell flat in this movie. It's about ninety minutes of one guy ri

## 2. Ποσοτικοποίηση “θορύβου” στο raw κείμενο

Πριν τον καθαρισμό, πραγματοποιείται ένας γρήγορος έλεγχος του περιεχομένου των reviews.
Σκοπός είναι να εκτιμηθεί πόσο συχνά εμφανίζονται μοτίβα που συνήθως επιβαρύνουν την προεπεξεργασία:

- URLs (σύνδεσμοι)
- HTML tags (π.χ. `<br />`)
- έντονη στίξη / punctuation
- ειδικοί χαρακτήρες
- υψηλή παρουσία stopwords

In [52]:
# Εντοπισμός reviews που περιέχουν URLs
import re

df['has_url'] = df['review'].str.contains(
    r'http\S+|www\S+',
    regex=True,
    na=False)

df['has_url'].value_counts()

has_url
False    99541
True       459
Name: count, dtype: int64

In [53]:
# Εντοπισμός reviews που περιέχουν HTML Tag
df['has_html'] = df['review'].str.contains(
    r'</?[a-zA-Z]+[^>]*>',
    regex=True,
    na=False)

df['has_html'].value_counts()

has_html
True     58813
False    41187
Name: count, dtype: int64

In [54]:
# Εντοπισμός reviews που περιέχουν punctuation
import string

punctuation = string.punctuation  # !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~

df['has_punctuation'] = df['review'].str.contains(
    f"[{re.escape(punctuation)}]",
    regex=True,
    na=False)

df['has_punctuation'].value_counts()

has_punctuation
True     99996
False        4
Name: count, dtype: int64

In [55]:
# Εντοπισμός reviews που περιέχουν ειδικούς χαρακτήρες
import re

df['has_special_chars'] = df['review'].str.contains(
    r'[^a-zA-Z0-9\s]',
    regex=True,
    na=False)

df['has_special_chars'].value_counts()

has_special_chars
True     99996
False        4
Name: count, dtype: int64

In [56]:
# Εντοπισμός reviews που περιέχουν stopwords
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Mkasi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [57]:
def count_stopwords(text):
    if pd.isna(text):
        return 0
    words = text.lower().split()
    return sum(1 for w in words if w in stop_words)

df['stopword_count'] = df['review'].apply(count_stopwords)

df['stopword_count'].describe()

count    100000.00000
mean        105.09492
std          77.82455
min           0.00000
25%          57.00000
50%          80.00000
75%         127.00000
max        1061.00000
Name: stopword_count, dtype: float64

In [58]:
df['has_stopwords'] = df['stopword_count'] > 0

df['has_stopwords'].value_counts()

has_stopwords
True     99999
False        1
Name: count, dtype: int64

In [59]:
from nltk.tokenize import WhitespaceTokenizer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
import nltk

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

w_tokenizer = WhitespaceTokenizer()
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Mkasi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Mkasi\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Mkasi\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


## 3. Καθαρισμός κειμένου & κανονικοποίηση 

Σε αυτό το στάδιο δημιουργείται η στήλη `clean_review`, η οποία αποτελεί την “κανονικοποιημένη” μορφή του review.
Το cleaning στοχεύει σε:

- αφαίρεση URLs και HTML tags
- μετατροπή σε lowercase
- αφαίρεση χαρακτήρων που δεν προσθέτουν πληροφορία (punctuation / special symbols)
- κανονικοποίηση κενών και tokenization
- (προαιρετικά) stopwords removal και lemmatization

In [60]:
negations = {"not", "no", "never"}
stop_words2 = stop_words - negations

def clean_and_lemmatize(text):
    if text is None or pd.isna(text):
        return ""

    text = str(text)

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', ' ', text)

    # Remove HTML tags
    text = re.sub(r'</?[a-zA-Z]+[^>]*>', ' ', text)

    # Lowercase
    text = text.lower()

    # Remove punctuation & special characters
    text = re.sub(r'[^a-z\s]', ' ', text)

    # Normalize spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Remove stopwords (EXCEPT negations) + lemmatization
    tokens = [
    lemmatizer.lemmatize(w, pos='v')
    for w in text.split()
    if (w not in stop_words2 and len(w) > 2) or w in negations]

    return " ".join(tokens)

In [61]:
df['clean_review'] = df['review'].apply(clean_and_lemmatize)

In [62]:
i = 3
print("ORIGINAL:\n", df['review'].iloc[i])
print("\nCLEAN:\n", df['clean_review'].iloc[i])

ORIGINAL:
 Not even the Beatles could write songs everyone liked, and although Walter Hill is no mop-top he's second to none when it comes to thought provoking action movies. The nineties came and social platforms were changing in music and film, the emergence of the Rapper turned movie star was in full swing, the acting took a back seat to each man's overpowering regional accent and transparent acting. This was one of the many ice-t movies i saw as a kid and loved, only to watch them later and cringe. Bill Paxton and William Sadler are firemen with basic lives until a burning building tenant about to go up in flames hands over a map with gold implications. I hand it to Walter for quickly and neatly setting up the main characters and location. But i fault everyone involved for turning out Lame-o performances. Ice-t and cube must have been red hot at this time, and while I've enjoyed both their careers as rappers, in my opinion they fell flat in this movie. It's about ninety minutes of 

## 4. Αναπαράσταση Κειμένου με TF-IDF

Η αναπαράσταση **Term Frequency – Inverse Document Frequency (TF-IDF)** μετατρέπει το κείμενο
σε sparse διανύσματα χαρακτηριστικών, αποτυπώνοντας τη σημασία κάθε όρου σε σχέση με το σύνολο των reviews.

### Επιλογή υπερ-παραμέτρων

Οι υπερ-παράμετροι του TF-IDF επιλέχθηκαν με στόχο την ισορροπία μεταξύ εκφραστικότητας και περιορισμού θορύβου:

- **`ngram_range = (1, 2)`**  
- **`min_df`**  
- **`max_df`**  
 - **`sublinear_tf = True`**  

In [63]:
df['clean_review'] = df['clean_review'].fillna("").astype(str)

In [64]:
df['clean_review'].head(10)

0    costner drag movie far longer necessary aside ...
1    example majority action film generic bore real...
2    first hate moronic rappers could act gun press...
3    not even beatles could write songs everyone li...
4    brass picture movies not fit word really somew...
5    funny thing happen watch mosquito one hand her...
6    german horror film one weirdest see not aware ...
7    long time fan japanese film expect really both...
8    tokyo eye tell year old japanese girl fall lik...
9    wealthy horse ranchers buenos air long stand n...
Name: clean_review, dtype: object

In [65]:
(df['clean_review'].str.len() == 0).sum()

np.int64(0)

In [66]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    ngram_range=(1,2),
    min_df=10,
    max_df=0.9,
    sublinear_tf=True)
X = tfidf.fit_transform(df['clean_review'])
print(X.shape)

(100000, 163495)


## 5. Word2Vec Embeddings & Αναπαράσταση Reviews

Σε αντίθεση με τις bag-of-words προσεγγίσεις (όπως το TF-IDF), το **Word2Vec** μαθαίνει
πυκνά (dense) διανύσματα λέξεων, στα οποία λέξεις με παρόμοιο σημασιολογικό περιεχόμενο
βρίσκονται κοντά στο ίδιο embedding space.

### Επιλογή υπερ-παραμέτρων Word2Vec

Οι υπερ-παράμετροι επιλέχθηκαν με στόχο την ισορροπία μεταξύ εκφραστικότητας,
γενίκευσης και υπολογιστικής αποδοτικότητας:

- **`vector_size = 200`**
- **`window = 5`**  
- **`min_count = 10`**  
- **`sg = 1` (Skip-gram)**  
- **`epochs = 10`**  

In [67]:
sentences = df['clean_review'].apply(lambda x: x.split()).tolist()

In [68]:
sentences[0][:10]

['costner',
 'drag',
 'movie',
 'far',
 'longer',
 'necessary',
 'aside',
 'terrific',
 'sea',
 'rescue']

In [69]:
!pip install gensim

In [70]:
import logging

# Καθαρισμός προηγούμενων ρυθμίσων logging ώστε να αποφύγουμε διπλές ή μπερδεμένες εκτυπώσεις στο output 
for h in logging.root.handlers[:]:
    logging.root.removeHandler(h)

logging.basicConfig(
    format="%(asctime)s : %(levelname)s : %(message)s",
    level=logging.INFO
)

# Διασφαλίζουμε ότι τα logs της βιβλιοθήκης gensim εμφανίζονται στο επίπεδο INFO
logging.getLogger("gensim").setLevel(logging.INFO)

In [71]:
from gensim.models import Word2Vec

w2v = Word2Vec(
    sentences=sentences,
    vector_size=200,
    window=5,
    min_count=10,
    sg=1,
    workers=4,
    epochs=10)

2026-02-08 23:15:36,844 : INFO : collecting all words and their counts
2026-02-08 23:15:36,846 : INFO : PROGRESS: at sentence #0, processed 0 words, keeping 0 word types
2026-02-08 23:15:37,441 : INFO : PROGRESS: at sentence #10000, processed 1149492 words, keeping 38572 word types
2026-02-08 23:15:37,960 : INFO : PROGRESS: at sentence #20000, processed 2343216 words, keeping 54661 word types
2026-02-08 23:15:38,502 : INFO : PROGRESS: at sentence #30000, processed 3496807 words, keeping 66310 word types
2026-02-08 23:15:39,095 : INFO : PROGRESS: at sentence #40000, processed 4697283 words, keeping 76393 word types
2026-02-08 23:15:39,576 : INFO : PROGRESS: at sentence #50000, processed 5915334 words, keeping 85897 word types
2026-02-08 23:15:40,223 : INFO : PROGRESS: at sentence #60000, processed 7137456 words, keeping 94376 word types
2026-02-08 23:15:40,869 : INFO : PROGRESS: at sentence #70000, processed 8337506 words, keeping 101883 word types
2026-02-08 23:15:41,281 : INFO : PROGR

In [72]:
w2v.save("imdb_w2v_200d_sg.model")

2026-02-08 23:29:25,296 : INFO : Word2Vec lifecycle event {'fname_or_handle': 'imdb_w2v_200d_sg.model', 'separately': 'None', 'sep_limit': 10485760, 'ignore': frozenset(), 'datetime': '2026-02-08T23:29:25.296238', 'gensim': '4.4.0', 'python': '3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]', 'platform': 'Windows-11-10.0.26200-SP0', 'event': 'saving'}
2026-02-08 23:29:25,298 : INFO : not storing attribute cum_table
2026-02-08 23:29:25,448 : INFO : saved imdb_w2v_200d_sg.model


In [73]:
from gensim.models import Word2Vec
w2v = Word2Vec.load("imdb_w2v_200d_sg.model")

2026-02-08 23:29:27,955 : INFO : loading Word2Vec object from imdb_w2v_200d_sg.model
2026-02-08 23:29:28,065 : INFO : loading wv recursively from imdb_w2v_200d_sg.model.wv.* with mmap=None
2026-02-08 23:29:28,067 : INFO : setting ignored attribute cum_table to None
2026-02-08 23:29:29,646 : INFO : Word2Vec lifecycle event {'fname': 'imdb_w2v_200d_sg.model', 'datetime': '2026-02-08T23:29:29.646250', 'gensim': '4.4.0', 'python': '3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]', 'platform': 'Windows-11-10.0.26200-SP0', 'event': 'loaded'}


In [74]:
w2v.wv.most_similar("movie", topn=10)
w2v.wv.most_similar("bad", topn=10)

[('terrible', 0.7330049872398376),
 ('horrible', 0.7081782817840576),
 ('awful', 0.6841441988945007),
 ('good', 0.6796376705169678),
 ('lousy', 0.6539463996887207),
 ('crappy', 0.6191955208778381),
 ('atrocious', 0.6046223044395447),
 ('poor', 0.5919785499572754),
 ('stupid', 0.5836169123649597),
 ('worse', 0.5743613243103027)]

In [75]:
w2v.wv.most_similar("good", topn=10)

[('great', 0.7376585602760315),
 ('decent', 0.6872087121009827),
 ('bad', 0.6796376705169678),
 ('excellent', 0.6207667589187622),
 ('fine', 0.6120879650115967),
 ('nice', 0.6052878499031067),
 ('really', 0.6019394993782043),
 ('awsome', 0.5787608623504639),
 ('well', 0.5774579644203186),
 ('okay', 0.5661510229110718)]

In [76]:
w2v.wv.similarity("good", "great")

np.float32(0.7376586)

Οι πιο κοντινές λέξεις που προκύπτουν για τους όρους *good* και *bad*
είναι σημασιολογικά συνεπείς και συναισθηματικά φορτισμένες,
γεγονός που υποδεικνύει ότι το Word2Vec έχει μάθει χρήσιμες
σημασιολογικές αναπαραστάσεις για το task του sentiment analysis.

In [77]:
import numpy as np

def document_vector(tokens, model):
    vectors = [model.wv[w] for w in tokens if w in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

In [78]:
X_w2v = np.vstack([
    document_vector(tokens, w2v)
    for tokens in sentences
])

X_w2v.shape

(100000, 200)

In [79]:
X_w2v.shape = (100000, 200)

## 6. Μείωση διάστασης με PCA

Τα Word2Vec document vectors μπορεί να έχουν σχετικά υψηλή διάσταση (π.χ. 200).
Εφαρμόζω PCA για να μειώσω τη διάσταση σε 50 components

In [80]:
from sklearn.decomposition import PCA

pca = PCA(n_components=50, random_state=42)
X_w2v_pca = pca.fit_transform(X_w2v)

X_w2v_pca.shape

(100000, 50)

In [81]:
df.columns

Index(['Unnamed: 0', 'type', 'review', 'label', 'file', 'has_url', 'has_html',
       'has_punctuation', 'has_special_chars', 'stopword_count',
       'has_stopwords', 'clean_review'],
      dtype='object')

In [82]:
# Κρατάμε αντίγραφο
df_model = df[['clean_review', 'label']].copy()

df_model['clean_review'] = df_model['clean_review'].fillna("").astype(str)

df_model.head()

,clean_review,label
0,costner drag movie far longer necessary aside ...,neg
1,example majority action film generic bore real...,neg
2,first hate moronic rappers could act gun press...,neg
3,not even beatles could write songs everyone li...,neg
4,brass picture movies not fit word really somew...,neg


In [83]:
df_model['label'].value_counts()

label
unsup    50000
neg      25000
pos      25000
Name: count, dtype: int64

In [84]:
df_model = df_model[df_model['label'].isin(['pos', 'neg'])].copy()

In [85]:
df_model['label'].value_counts()

label
neg    25000
pos    25000
Name: count, dtype: int64

In [86]:
# Φτιάχνουμε binary labels
df_model = df[['clean_review', 'label']].copy()

# Αντικαθιστούμε πιθανά NaN reviews με κενό string
df_model['clean_review'] = df_model['clean_review'].fillna("").astype(str)

# Περιοριζόμαστε σε καθαρό binary πρόβλημα (pos / neg)
df_model = df_model[df_model['label'].isin(['pos', 'neg'])].copy()

# Μετατροπή των labels σε δυαδική μορφή
df_model['y'] = df_model['label'].map({'neg': 0, 'pos': 1}).astype(int)

X = df_model['clean_review']
y = df_model['y']

# Χωρίζουμε τα δεδομένα σε train και test σύνολα διατηρώντας την ίδια αναλογία θετικών/αρνητικών κλάσεων
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# TF-IDF + Logistic Regression
pipe = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.95,
        sublinear_tf=True,
        max_features=30000
    )),
    ("clf", LogisticRegression(
        max_iter=2000,
        solver="liblinear",
        C=2.0
    ))
])

# Εκπαίδευση μοντέλου
pipe.fit(X_train, y_train)

# Predict + Evaluation
# Αξιολόγηση στο test set
# Προβλέψεις στο test σύνολο
y_pred = pipe.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred, target_names=["neg", "pos"]))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.9102

Classification report:
               precision    recall  f1-score   support

         neg       0.92      0.90      0.91      5000
         pos       0.90      0.92      0.91      5000

    accuracy                           0.91     10000
   macro avg       0.91      0.91      0.91     10000
weighted avg       0.91      0.91      0.91     10000


Confusion matrix:
 [[4500  500]
 [ 398 4602]]


In [87]:
!pip install xgboost

In [88]:
from xgboost import XGBClassifier

In [89]:
xgb_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=30000,
        ngram_range=(1, 2),
        stop_words="english"
    )),
    ("clf", XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ))
])

In [90]:
xgb_pipe.fit(X_train, y_train)

y_pred_xgb = xgb_pipe.predict(X_test)

print("XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))

XGBoost Accuracy: 0.8653
              precision    recall  f1-score   support

           0       0.89      0.84      0.86      5000
           1       0.85      0.89      0.87      5000

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000



In [91]:
df_model = df[['clean_review', 'label']].copy()
df_model['clean_review'] = df_model['clean_review'].fillna("").astype(str)

df_model = df_model[df_model['label'].isin(['pos', 'neg'])].copy()

df_model['y'] = df_model['label'].map({'neg': 0, 'pos': 1}).astype(int)

X = df_model['clean_review']
y = df_model['y']


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


pipe_log = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.95,
        sublinear_tf=True,
        max_features=30000
    )),
    ("clf", LogisticRegression(
        max_iter=2000,
        solver="liblinear",
        C=2.0
    ))
])


pipe_log.fit(X_train, y_train)

y_pred = pipe_log.predict(X_test)

print("=== Test Set Evaluation ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["neg", "pos"]))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


# Cross-Validation 

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores = cross_validate(
    pipe_log,
    X_train,
    y_train,
    cv=cv,
    scoring=["accuracy", "precision", "recall", "f1"],
    n_jobs=-1
)

print("\n=== Cross-Validation Results (5-fold) ===")
print("Accuracy : {:.4f} ± {:.4f}".format(
    cv_scores["test_accuracy"].mean(),
    cv_scores["test_accuracy"].std()
))
print("Precision: {:.4f} ± {:.4f}".format(
    cv_scores["test_precision"].mean(),
    cv_scores["test_precision"].std()
))
print("Recall   : {:.4f} ± {:.4f}".format(
    cv_scores["test_recall"].mean(),
    cv_scores["test_recall"].std()
))
print("F1-score : {:.4f} ± {:.4f}".format(
    cv_scores["test_f1"].mean(),
    cv_scores["test_f1"].std()
))

=== Test Set Evaluation ===
Accuracy: 0.9102

Classification Report:
              precision    recall  f1-score   support

         neg       0.92      0.90      0.91      5000
         pos       0.90      0.92      0.91      5000

    accuracy                           0.91     10000
   macro avg       0.91      0.91      0.91     10000
weighted avg       0.91      0.91      0.91     10000


Confusion Matrix:
[[4500  500]
 [ 398 4602]]

=== Cross-Validation Results (5-fold) ===
Accuracy : 0.9033 ± 0.0022
Precision: 0.8966 ± 0.0042
Recall   : 0.9117 ± 0.0027
F1-score : 0.9041 ± 0.0020


In [ ]:
df_model = df[['clean_review', 'label']].copy()
df_model['clean_review'] = df_model['clean_review'].fillna("").astype(str)

df_model = df_model[df_model['label'].isin(['pos', 'neg'])].copy()

df_model['y'] = df_model['label'].map({'neg': 0, 'pos': 1}).astype(int)

X = df_model['clean_review']
y = df_model['y']


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


pipe_xgb = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.95,
        sublinear_tf=True,
        max_features=80000
    )),
    ("xgb", XGBClassifier(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        n_jobs=-1,
        random_state=42
    ))
])


pipe_xgb.fit(X_train, y_train)

y_pred = pipe_xgb.predict(X_test)

print("=== XGBoost Test Set Evaluation ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["neg", "pos"]))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


# Cross-Validation 

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores_xgb = cross_validate(
    pipe_xgb,
    X,
    y,
    cv=cv,
    scoring=["accuracy", "precision", "recall", "f1"],
    n_jobs=-1
)

print("\n=== XGBoost Cross-Validation Results (5-fold) ===")
print("Accuracy : {:.4f} ± {:.4f}".format(
    cv_scores_xgb["test_accuracy"].mean(),
    cv_scores_xgb["test_accuracy"].std()
))
print("Precision: {:.4f} ± {:.4f}".format(
    cv_scores_xgb["test_precision"].mean(),
    cv_scores_xgb["test_precision"].std()
))
print("Recall   : {:.4f} ± {:.4f}".format(
    cv_scores_xgb["test_recall"].mean(),
    cv_scores_xgb["test_recall"].std()
))
print("F1-score : {:.4f} ± {:.4f}".format(
    cv_scores_xgb["test_f1"].mean(),
    cv_scores_xgb["test_f1"].std()
))